# Phi-4-mini SFT 파인튜닝 (재설계본)
담당: Generation 파트 (한의정)

## 이전 FT가 망가진 원인 → 이번 수정
1. **거절 예시 0개** → 무관 context + "제공된 문서에서 확인할 수 없습니다" **거절 샘플 25% 추가**
2. **청크 복붙 요약(text[:300])** → 폐기. 근거 기반 QA 형태로 재구성
3. **학습 포맷 ≠ 추론 포맷** → `generation.py`의 `BASE_SYSTEM_PROMPT`·`USER_PROMPT_TEMPLATE` **그대로 사용** + `apply_chat_template`으로 토큰화(수제 토큰 금지)
4. **lr 2e-4·3ep 과적합** → **lr 1e-4·2ep** 완화
5. **eval_retrieval_579 누수** → 평가셋이므로 학습에서 제외

> 검증: 재학습 후 chunks_all-FT 평가에서 rejection·faithfulness가 BASE 수준(3.5/3.2)으로 회복되는지 확인.


In [ ]:
# [0] 설치
!pip install -q --upgrade transformers peft trl accelerate datasets sentencepiece
!rm -rf ~/.cache/huggingface/modules/transformers_modules/microsoft/Phi-4-mini-instruct 2>/dev/null
print("설치 완료 — 런타임 재시작 메시지 뜨면 재시작 후 [1]부터")


In [ ]:
# [1] 사전 체크 — GPU/경로/데이터 신선도 (꼼꼼체크)
import os, torch, json, glob
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

assert torch.cuda.is_available(), '❌ GPU 없음 — 런타임 유형을 GPU로'
total_vram = torch.cuda.get_device_properties(0).total_memory/1024**3
gpu_name   = torch.cuda.get_device_name(0)
print(f'GPU: {gpu_name} | VRAM: {total_vram:.1f}GB')

# ★ VRAM 따라 BATCH 자동 조정
if   total_vram >= 70: BATCH_SIZE, GRAD_ACCUM = 8, 2   # H100/A100-80G
elif total_vram >= 38: BATCH_SIZE, GRAD_ACCUM = 4, 4   # A100-40G/L40
elif total_vram >= 20: BATCH_SIZE, GRAD_ACCUM = 2, 8   # L4/3090
else:                  BATCH_SIZE, GRAD_ACCUM = 1, 16  # T4 16G
print(f'BATCH 자동조정: batch={BATCH_SIZE} grad_accum={GRAD_ACCUM} (eff={BATCH_SIZE*GRAD_ACCUM})')

DRIVE = '/content/drive/MyDrive/data/bidmate'
CHUNK_JSON_PATH = f'{DRIVE}/kh_v3.json'       # 청킹 — 평가와 동일 청킹 권장
EVAL_CSV_PATH   = f'{DRIVE}/eval/eval_retrieval_579.csv'  # ★ 학습 제외(평가셋), 거절패턴 참고만
OUTPUT_DIR      = f'{DRIVE}/peft_output/phi4-mini-v2'      # ★ 기존 어댑터 덮어쓰지 않게 v2
CODE_DIR        = f'{DRIVE}/code'

# 필수 파일 존재 확인
for label,p in [('청크JSON',CHUNK_JSON_PATH),('eval CSV',EVAL_CSV_PATH),('code폴더',CODE_DIR)]:
    print(f'{"OK" if os.path.exists(p) else "MISSING":8}{label:10}{p}')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 기존 어댑터 존재 시 경고 (데이터 신선도)
old = f'{DRIVE}/peft_output/phi4-mini/lora_adapter'
if os.path.exists(old):
    print(f'⚠️ 기존 어댑터 존재: {old} — 이번 학습은 v2 폴더에 저장되어 덮어쓰지 않음')


In [ ]:
# [2] 추론 프롬프트 임포트 — ★ generation.py 와 100% 동일 포맷 사용
#   학습/추론 불일치 제거의 핵심. generation.py 의 프롬프트를 직접 가져온다.
import sys
if CODE_DIR not in sys.path: sys.path.insert(0, CODE_DIR)

# generation.py 가 config 를 import 하므로 최소 stub 주입
import types
if 'config' not in sys.modules:
    _c = types.ModuleType('config')
    for k,v in dict(ADAPTER_PATH='', BASE_MODEL_ID='microsoft/Phi-4-mini-instruct',
                    LLM_MODEL='microsoft/Phi-4-mini-instruct', MAX_TOKENS_GENERATE=800,
                    MAX_TOKENS_REWRITE=300).items():
        setattr(_c, k, v)
    sys.modules['config'] = _c

import importlib.util
spec = importlib.util.spec_from_file_location('generation', f'{CODE_DIR}/generation.py')
G = importlib.util.module_from_spec(spec)
try:
    spec.loader.exec_module(G)
    BASE_SYSTEM_PROMPT   = G.BASE_SYSTEM_PROMPT
    TYPE_INSTRUCTIONS    = G.TYPE_INSTRUCTIONS
    USER_PROMPT_TEMPLATE = G.USER_PROMPT_TEMPLATE
    print('✅ generation.py 에서 프롬프트 임포트 성공 (추론과 동일 포맷 보장)')
except Exception as e:
    print('⚠️ generation.py 임포트 실패 — 하드코딩 폴백 사용:', e)
    BASE_SYSTEM_PROMPT = ("당신은 공공 입찰 RFP 문서 분석 어시스턴트입니다.\n아래 규칙을 반드시 지키세요.\n\n"
        "규칙1: [검색된 문서]에 있는 내용만 답변하세요. 문서 외 지식 사용 금지.\n"
        "규칙2: 금액/날짜/기간 등 수치는 문서에 나온 숫자 그대로만 쓰세요. 계산하거나 변환하지 마세요.\n"
        "규칙3: 문서에 답이 없으면 반드시 \"제공된 문서에서 확인할 수 없습니다\"라고만 답하세요.\n"
        "규칙4: 문서에 없는 내용을 추측하거나 지어내는 것은 절대 금지입니다.\n"
        "규칙5: 답변은 간결하게 핵심만 작성하세요.\n")
    TYPE_INSTRUCTIONS = {"single": "\n[답변 형식]\n- 질문에 직접 답변하는 1~3문장으로 시작합니다.\n"}
    USER_PROMPT_TEMPLATE = "[검색된 문서]\n{context}\n\n---\n\n[질문]\n{query}"

REJECT_ANSWER = "제공된 문서에서 확인할 수 없습니다."   # ★ 추론 system 규칙3 문구와 정확히 일치
print('거절 정답 문구:', REJECT_ANSWER)


In [ ]:
# [3] 데이터 구성 — 근거기반 QA + 거절 샘플 (포맷=추론과 동일)
import json, random, pandas as pd
from collections import defaultdict
random.seed(42)

CHUNK_MIN_LEN  = 100
REJECT_RATIO   = 0.25     # ★ 거절 샘플 비중

with open(CHUNK_JSON_PATH, encoding='utf-8') as f:
    chunks = json.load(f)
print(f'총 청크: {len(chunks):,}')

def clean(t): return t.replace('[TABLE]','').replace('[IMAGE]','').strip()
def valid(t): return len(clean(t)) >= CHUNK_MIN_LEN

by_agency = defaultdict(list)
for c in chunks:
    ag = c.get('metadata',{}).get('agency','')
    if ag and valid(c['text']): by_agency[ag].append(c)
valid_chunks = [c for c in chunks if valid(c['text'])]
print(f'유효 청크: {len(valid_chunks):,} | 기관 수: {len(by_agency)}')

# ── eval CSV: 학습엔 직접 안 쓰고, GT-QA 형식만 참고 (누수 방지) ──
# 대신 청크에서 "근거 안에 답이 있는" positive QA 를 합성한다.
def make_user(context, query):
    return USER_PROMPT_TEMPLATE.format(context=context, query=query)

samples = []

# (A) Positive QA — 청크 내용을 묻고, 청크 근거로 답 (요약 복붙 아님: 질의-응답 형태)
for c in valid_chunks:
    meta = c.get('metadata',{})
    ag   = meta.get('agency','해당 기관')
    proj = meta.get('project_name', meta.get('source_file','해당 사업'))
    txt  = clean(c['text'])
    ctx  = f"[1] {txt[:800]}\n(출처: {ag})"
    q    = f"{ag}의 '{proj}' 관련 내용을 문서를 근거로 설명해 주세요."
    # ⚠️ 한계: 규칙기반 합성이라 응답이 청크 첫 문장 인용에 가깝다(완전한 복붙 회피는 불가).
    #   → 품질을 더 높이려면 이 블록을 LLM 합성으로 교체 권장:
    #     청크를 gpt-4o-mini 등에 주고 "이 문서 근거로 질문-답변 쌍 생성" → response 에 사용.
    #   지금은 '문서 근거 명시 + 핵심 인용' 형태로 최소한의 근거기반 답변을 만든다.
    sents = [x.strip() for x in txt.replace('\n',' ').split('.') if len(x.strip())>10]
    core  = '. '.join(sents[:2])[:240]
    a     = f"문서에 근거하면 다음과 같습니다. {core}." if core else f"문서에 따르면 {ag}의 해당 사업 관련 내용이 기재되어 있습니다."
    samples.append({'system': BASE_SYSTEM_PROMPT + TYPE_INSTRUCTIONS.get('single',''),
                    'user': make_user(ctx, q), 'assistant': a})

random.shuffle(samples)
POS_N = min(3000, len(samples))   # positive 상한
samples = samples[:POS_N]
print(f'positive QA: {len(samples)}')

# (B) Rejection 샘플 — 질문과 무관한 청크를 context 로 주고 거절하도록 학습
agencies = list(by_agency.keys())
n_reject = int(len(samples) * REJECT_RATIO / (1-REJECT_RATIO))
reject = []
for _ in range(n_reject):
    a1, a2 = random.sample(agencies, 2)          # 서로 다른 두 기관
    wrong_ctx_chunk = random.choice(by_agency[a1])
    ctx = f"[1] {clean(wrong_ctx_chunk['text'])[:800]}\n(출처: {a1})"
    # a2 를 묻는데 context 는 a1 → 근거 없음 → 거절해야 정답
    q = f"{a2}의 사업 예산 규모는 얼마입니까?"
    reject.append({'system': BASE_SYSTEM_PROMPT + TYPE_INSTRUCTIONS.get('single',''),
                   'user': make_user(ctx, q), 'assistant': REJECT_ANSWER})
print(f'rejection 샘플: {len(reject)}')

all_samples = samples + reject
random.shuffle(all_samples)
print(f'총 학습 샘플: {len(all_samples)} (거절 비중 {len(reject)/len(all_samples)*100:.0f}%)')


In [ ]:
# [4] 모델/토크나이저 로드 + apply_chat_template 으로 텍스트화 (★ 추론과 동일 토큰화)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from datasets import Dataset

MODEL_ID = 'microsoft/Phi-4-mini-instruct'
MAX_SEQ_LEN = 2048

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = 'right'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = MAX_SEQ_LEN

def to_text(s):
    # ★ 학습 텍스트를 추론과 똑같은 apply_chat_template 으로 생성 (수제 토큰 금지)
    chat = [{'role':'system','content':s['system']},
            {'role':'user','content':s['user']},
            {'role':'assistant','content':s['assistant']}]
    return {'text': tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=False)}

ds = Dataset.from_list([to_text(s) for s in all_samples])
ds = ds.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = ds['train'], ds['test']
print(f'학습 {len(train_ds)} | 검증 {len(eval_ds)}')
print('\n샘플 텍스트(앞 400자):\n', train_ds[0]['text'][:400])

# 모델 (bf16; T4면 fp16 권장)
use_bf16 = torch.cuda.get_device_capability(0)[0] >= 8   # Ampere+ 만 bf16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    device_map='auto')
model.config.use_cache = False

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('bf16' if use_bf16 else 'fp16', '학습')


In [ ]:
# [5] 학습 — lr 1e-4 · 2ep (과적합 완화), eval_loss 기준 best 저장
from trl import SFTTrainer, SFTConfig

LEARNING_RATE = 1e-4    # ★ 2e-4 → 1e-4
NUM_EPOCHS    = 2       # ★ 3 → 2
use_bf16 = torch.cuda.get_device_capability(0)[0] >= 8

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    bf16=use_bf16, fp16=not use_bf16, tf32=use_bf16,
    logging_steps=10,
    eval_strategy='steps', eval_steps=100,
    save_strategy='steps', save_steps=100, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
    max_length=MAX_SEQ_LEN,
    report_to='none', dataloader_num_workers=2,
)
trainer = SFTTrainer(model=model, args=args,
                     train_dataset=train_ds, eval_dataset=eval_ds,
                     processing_class=tokenizer)
trainer.train()
print('학습 완료')


In [ ]:
# [6] 어댑터 저장 (v2)
SAVE_DIR = f'{OUTPUT_DIR}/lora_adapter'
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
import os
print('저장:', SAVE_DIR)
print('파일:', os.listdir(SAVE_DIR))
print('\n★ 평가 시 config 의 ADAPTER_PATH 를 이 경로로 교체:')
print(f"   cfg.ADAPTER_PATH=Path('{SAVE_DIR}')")


In [ ]:
# [7] 추론 스모크 테스트 — 거절이 실제로 되는지 확인
import torch
model.eval()

def gen(system, user):
    chat=[{'role':'system','content':system},{'role':'user','content':user}]
    ids=tokenizer.apply_chat_template(chat, return_tensors='pt', add_generation_prompt=True).to(model.device)
    with torch.no_grad():
        out=model.generate(ids, max_new_tokens=200, do_sample=False,
                           pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

sysp = BASE_SYSTEM_PROMPT + TYPE_INSTRUCTIONS.get('single','')
# (1) 근거 있는 질문
ctx_ok = "[1] 한국가스공사 차세대 ERP 구축 사업, 예산 14,107,009,000원.\n(출처: 한국가스공사)"
print('● 근거있음 →', gen(sysp, USER_PROMPT_TEMPLATE.format(context=ctx_ok, query="한국가스공사 ERP 사업 예산은?")))
# (2) 근거 없는 질문 → 거절해야 정상
ctx_no = "[1] 서울교통공사 승강장 안전문 정비 용역.\n(출처: 서울교통공사)"
print('● 근거없음 →', gen(sysp, USER_PROMPT_TEMPLATE.format(context=ctx_no, query="한국전력공사 사업 예산은?")))
print('\n→ (2)가 "제공된 문서에서 확인할 수 없습니다" 류로 나오면 거절 학습 성공')
